# Text as numbers: tokens, embeddings and transformers

Machine Learning Engineer track, lesson MLE7. Part of [datafromzero](https://datafromzero.github.io), data and text CC BY 4.0. The text is the site's own: eight lessons of the SQL stage, also CC BY 4.0.

**Goals:** see text become tokens, turn tokens into embeddings and compare meaning with cosine similarity, see what attention does to a word's vector, and fit a TF-IDF baseline before a pretrained encoder.

**How to use this notebook:** run the cells from top to bottom. When in doubt, **Run all from the top**: restart the session and run every cell from the first.
Every exercise has a cell for your code and a check cell that compares your answer with a known answer. Hints and solutions are folded away
under each exercise, free to open. The working cell of Exercise 2 starts empty: you write it. A CPU runtime is enough; the first run downloads a small model (about 90 MB).

## Setup

In [ ]:
import importlib.metadata
import math
import numbers
import re
import subprocess
import sys
import urllib.request

PINS = ["transformers==5.24.1", "sentence-transformers==6.3.0"]               # the versions this notebook was run with
for pin in PINS:
    package, version = pin.split("==")
    try:
        installed = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        installed = None
    if installed != version:                                                  # install the pinned version when Colab has another
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pin], check=True)

import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, train_test_split
from transformers import AutoModel

SEED = 2026       # the seed of the whole track
torch.set_num_threads(1)       # one thread: the sums are added in the same order on every run

EXERCISES = ['Exercise 1', 'Exercise 2', 'Exercise 3']


# Run this cell as it is. It also holds the answer checker used by the exercise cells.
# Run all from the top: restart the session, run every cell from the first, then read the last cell.

RESULTS = {}


def same(got, expected, tol=1e-9):
    """True when two values agree: floats within `tol`, lists and dicts item by item."""
    try:
        if isinstance(expected, dict):
            return isinstance(got, dict) and got.keys() == expected.keys() and all(same(got[k], expected[k], tol) for k in expected)
        if isinstance(expected, (list, tuple)):
            return isinstance(got, (list, tuple)) and len(got) == len(expected) and all(same(g, e, tol) for g, e in zip(got, expected))
        if isinstance(expected, bool) or isinstance(got, bool):
            return type(got) is type(expected) and got == expected
        if isinstance(expected, float):
            return isinstance(got, numbers.Real) and math.isclose(got, expected, rel_tol=1e-9, abs_tol=tol)
        if isinstance(expected, int):
            return isinstance(got, numbers.Integral) and got == expected
        return type(got) is type(expected) and got == expected
    except Exception:
        return False


def check(label, name, expected, traps=None, tol=1e-9):
    """Compare the variable called `name` with the known answer. It never stops Run all."""
    got = globals().get(name)
    if got is None:
        RESULTS[label] = None
        print(f"{label}: give {name} a value, then run this cell again.")
        return False
    if same(got, expected, tol):
        RESULTS[label] = True
        print(f"{label}: correct.")
        return True
    RESULTS[label] = False
    print(f"{label}: not yet. Expected {expected!r}, but {name} is {got!r}.")
    for wrong, message in (traps.items() if isinstance(traps, dict) else (traps or [])):
        if same(got, wrong, tol):
            print("Hint:", message)
    return False

## The corpus

Eight lessons of the SQL stage, read from the release tag `v1.0.0` (or from this repository's copy of the same files). `paragraphs` keeps the prose: it drops the front matter, figures, code, tables and the exercise sections, and every block of fewer than 8 words. `paras` pairs each paragraph with its lesson; the lesson is the label this notebook predicts. `PROBE` is 16 questions written for this lesson, two for each lesson, each with the lesson that answers it: they are never used to fit anything. Run both cells as they are.

In [ ]:
# The corpus: eight published lessons of the SQL stage (CC BY 4.0), read from the release tag v1.0.0, or from this repository's own copy of the same files.
TEXT_BASE = "https://raw.githubusercontent.com/datafromzero/datafromzero.github.io/v1.0.0/src/content/docs/sql/"
TEXT_LOCAL = "../../src/content/docs/sql/"      # the same files, when this notebook runs inside a clone of the repository
LESSONS = ["null-handling", "group-by-having", "inner-join", "left-join-anti-join", "window-functions", "ctes", "dates-and-time", "case-expressions"]
SKIP_SECTIONS = {"Warm-up", "Finish it", "On your own", "Check yourself"}      # recall prompts and exercises are not teaching text


def read_lesson(slug):
    """The raw text of one lesson file: downloaded from the tag, or the local copy."""
    try:
        with urllib.request.urlopen(TEXT_BASE + slug + ".mdx", timeout=30) as reply:
            return reply.read().decode("utf-8")
    except Exception:
        with open(TEXT_LOCAL + slug + ".mdx", encoding="utf-8") as handle:
            return handle.read()


def paragraphs(raw):
    """The prose paragraphs of a lesson as (section heading, text): no front matter, figures, code, tables or exercise sections."""
    raw = re.sub(r"\A---\n.*?\n---\n", "", raw, flags=re.S)
    raw = re.sub(r"<figure.*?</figure>|<svg.*?</svg>|<RunSQL.*?/>|```.*?```", "", raw, flags=re.S)
    found, section = [], ""
    for block in re.split(r"\n\s*\n", raw):
        block = block.strip()
        if block.startswith("#"):
            section = block.lstrip("#").strip()
        elif block and section not in SKIP_SECTIONS and not block.startswith(("<", "|", "{/*")):
            block = re.sub(r"\[([^\]]+)\]\([^)]+\)", r"\1", re.sub(r"<[^>]+>", "", block)).replace("*", "").replace("`", "")
            block = re.sub(r"\s+", " ", block).strip()
            if len(block.split()) >= 8:                                           # a few words are a label, not a paragraph
                found.append((section, block))
    return found


CORPUS = {slug: paragraphs(read_lesson(slug)) for slug in LESSONS}
paras = [(slug, text) for slug in LESSONS for _, text in CORPUS[slug]]            # (lesson, paragraph) for every paragraph
texts, labels = [text for _, text in paras], [slug for slug, _ in paras]
print(len(paras), "paragraphs,", sum(len(text.split()) for text in texts), "words,", len(LESSONS), "lessons")

In [ ]:
PROBE = [
    ('Why does WHERE customer_id = NULL never return a row?', 'null-handling'),
    ('Which function replaces a missing value with a default?', 'null-handling'),
    ('What is the difference between WHERE and HAVING?', 'group-by-having'),
    ('How many orders does store 2 have in all?', 'group-by-having'),
    ('What happens to a row that has no partner in an inner join?', 'inner-join'),
    ('If a key repeats on both sides, how many rows does a join return?', 'inner-join'),
    ('How do I find customers who never placed an order?', 'left-join-anti-join'),
    ('Why did my LEFT JOIN lose the unmatched rows after I added a WHERE on the right table?', 'left-join-anti-join'),
    ('How can I add a total beside every row without losing rows?', 'window-functions'),
    ('Can I filter on the result of a window function in WHERE?', 'window-functions'),
    ('What is a CTE?', 'ctes'),
    ('Does a CTE stay in the database after the query ends?', 'ctes'),
    ('How do I get the first day of the month of a timestamp?', 'dates-and-time'),
    ('Why does BETWEEN miss orders placed on the last day of a month?', 'dates-and-time'),
    ('What does CASE return when no WHEN is true and there is no ELSE?', 'case-expressions'),
    ('Which WHEN wins when two tests are both true?', 'case-expressions'),
]
print(len(PROBE), "questions")

## The model

A small pretrained model that turns a sentence or a short paragraph into 384 numbers: `all-MiniLM-L6-v2` (22.7 million numbers, Apache 2.0), pinned to one revision of the model repository. Run this cell as it is.

In [ ]:
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
REVISION = "1110a243fdf4706b3f48f1d95db1a4f5529b4d41"                        # the exact version of the model files
model = SentenceTransformer(MODEL_NAME, revision=REVISION, device="cpu")
print(sum(p.numel() for p in model.parameters()))

## Unit 1: Tokens, not words

**Predict first.** A model reads tokens, not words. Each sentence below has a number of words (split on spaces). How many tokens will the tokenizer of the model make of each, and which sentence gets the most tokens for its words?

In [ ]:
tokenizer = model.tokenizer
for sentence in ["How many orders did store 2 take?", "COALESCE replaces a NULL with a default.", "date_trunc('month', ordered_at)"]:
    pieces = tokenizer.tokenize(sentence)
    print(len(sentence.split()), len(pieces), pieces)

## Unit 2: Embeddings and cosine similarity

**Predict first.** The first sentence below asks for the rows that have no match. Rank the other three by how close the model puts them to it: a question about the same thing in other words, a question that shares many words with it but is about something else, and an unrelated question. Will the sentence that shares the most words come first?

In [ ]:
sentences = ["How do I keep rows that have no match?", "Which join keeps unmatched rows?", "How do I keep my phone from losing its match history?", "What is the capital of Australia?"]
vectors = model.encode(sentences)
print(vectors.shape, [round(float(np.linalg.norm(v)), 3) for v in vectors])
print([round(float(score), 3) for score in vectors[0] @ vectors[1:].T])

## Unit 3: What attention does to a word

**Predict first.** The word `key` stands at the same place in both sentences below, so the model starts from the same vector for it. How similar, by cosine, are its two vectors at the input and after each of the six layers? And where does `key` look, in the last layer?

In [ ]:
encoder = AutoModel.from_pretrained(MODEL_NAME, revision=REVISION, attn_implementation="eager").eval()    # the same network, set to report its attention
pair = ["A table needs a primary key.", "To run it press the key."]
batch = tokenizer(pair, return_tensors="pt", padding=True)
with torch.no_grad():
    result = encoder(**batch, output_hidden_states=True, output_attentions=True)
tokens = tokenizer.convert_ids_to_tokens(batch["input_ids"][0])
where = tokens.index("key")                                                   # the place of `key`: the same in both sentences
print(tokens, where, encoder.config.num_hidden_layers, encoder.config.num_attention_heads)
print([round(float(torch.cosine_similarity(layer[0, where], layer[1, where], dim=0)), 3) for layer in result.hidden_states])
looks = result.attentions[-1][0].mean(0)[where]                              # last layer, mean of the 12 heads: how much `key` looks at each token
print(round(float(looks.sum()), 3), [(t, round(float(w), 3)) for t, w in sorted(zip(tokens, looks.tolist()), key=lambda item: -item[1])[:3]])

## Unit 4: A baseline first, then a pretrained encoder

**Predict first.** Both models read the same training paragraphs and predict which of the 8 lessons a paragraph comes from. A TF-IDF baseline (counts of words and word pairs) and the pretrained encoder (384 numbers for each paragraph) each feed a logistic regression. On the held-out paragraphs and on the 16 questions, which scores higher? A flat guess always names the most common lesson.

In [ ]:
train_text, test_text, train_lesson, test_lesson = train_test_split(texts, labels, test_size=0.25, random_state=SEED, stratify=labels)
questions, question_lesson = [q for q, _ in PROBE], [lesson for _, lesson in PROBE]
grid = {"C": [1, 10, 100]}                                                    # each head picks its C by 5-fold cross-validation on the training paragraphs

tfidf = TfidfVectorizer(sublinear_tf=True, ngram_range=(1, 2)).fit(train_text)
baseline = GridSearchCV(LogisticRegression(max_iter=3000), grid, cv=5).fit(tfidf.transform(train_text), train_lesson)
vectors_train, vectors_test, vectors_q = model.encode(train_text), model.encode(test_text), model.encode(questions)
encoder_head = GridSearchCV(LogisticRegression(max_iter=3000), grid, cv=5).fit(vectors_train, train_lesson)

flat = max(sorted(set(train_lesson)), key=train_lesson.count)       # ties go to the first name in alphabetical order
print(len(train_text), len(test_text), flat)
for kind, on_test, on_questions in [("flat guess", [flat] * len(test_text), [flat] * len(questions)),
                                    ("TF-IDF and logistic regression", baseline.predict(tfidf.transform(test_text)), baseline.predict(tfidf.transform(questions))),
                                    ("encoder and logistic regression", encoder_head.predict(vectors_test), encoder_head.predict(vectors_q))]:
    print(kind, round(float(np.mean(np.array(on_test) == np.array(test_lesson))), 3), int(np.sum(np.array(on_questions) == np.array(question_lesson))), "of", len(questions))

A check on the headline: 20 other splits of the paragraphs (optional, about two minutes). Each split uses the same 220 paragraphs, split another way.

In [ ]:
import statistics
import warnings

warnings.filterwarnings("ignore")
vectors_all = model.encode(texts)
rows = []
for seed in range(100, 120):
    a_text, b_text, a_lesson, b_lesson, a_vec, b_vec = train_test_split(texts, labels, vectors_all, test_size=0.25, random_state=seed, stratify=labels)
    tf = TfidfVectorizer(sublinear_tf=True, ngram_range=(1, 2)).fit(a_text)
    base = GridSearchCV(LogisticRegression(max_iter=3000), grid, cv=5).fit(tf.transform(a_text), a_lesson)
    head = GridSearchCV(LogisticRegression(max_iter=3000), grid, cv=5).fit(a_vec, a_lesson)
    rows.append((float(np.mean(base.predict(tf.transform(b_text)) == np.array(b_lesson))), float(np.mean(head.predict(b_vec) == np.array(b_lesson))),
                 int(np.sum(base.predict(tf.transform(questions)) == np.array(question_lesson))), int(np.sum(head.predict(model.encode(questions)) == np.array(question_lesson)))))
for kind, column in [("TF-IDF", 0), ("encoder", 1)]:
    print(kind, "paragraphs: median", round(statistics.median(r[column] for r in rows), 3), "range", round(min(r[column] for r in rows), 3), round(max(r[column] for r in rows), 3),
          "| questions: median", statistics.median(r[column + 2] for r in rows), "range", min(r[column + 2] for r in rows), max(r[column + 2] for r in rows))
print("encoder higher in", sum(r[1] > r[0] for r in rows), "of 20 splits, lower in", sum(r[1] < r[0] for r in rows))

## Finish it

### Exercise 1: Finish it

**Task:** **Finish it** Count the pieces of a SQL sentence. Take the sentence below, make its tokens with the tokenizer of unit 1, and put two numbers in the list `answer_1`: how many tokens it has, and how many of them are continuation pieces, the ones that start with `##`.

**Expected:** `answer_1` is a list of two whole numbers.

In [ ]:
sentence = "Customer 141 placed orders 2673 and 2744."
pieces = tokenizer.tokenize(sentence)                                        # given
answer_1 = None  # replace None with [number of tokens, number of pieces that start with ##]

In [ ]:
_ = check("Exercise 1", "answer_1", [10, 2], traps={(7, 0): "That counts words split on spaces, not tokens."})

<details>
<summary>Hint</summary>

A continuation piece is a string that starts with the two characters `##`. Count them with a loop or with `str.startswith`.

</details>

<details>
<summary>Half-solution</summary>

```python
answer_1 = [len(pieces), sum(piece.startswith("##") for piece in ...)]
```

</details>

<details>
<summary>Solution</summary>

```python
sentence ="Customer 141 placed orders 2673 and 2744."
pieces = tokenizer.tokenize(sentence)
answer_1 = [len(pieces), sum(piece.startswith("##") for piece in pieces)]
```

[10, 2]: the sentence has 7 words and 10 tokens. The tokenizer keeps `customer`, `placed` and `orders` whole and cuts the numbers `2673` and `2744` into `267`, `##3` and `274`, `##4`, so two pieces start with `##`. A long number costs several tokens.

</details>

## On your own

### Exercise 2: On your own

**Task:** **On your own** Before you start, say in one sentence what you expect: does a question land next to a paragraph of its own lesson more often than the 1 in 8 of a blind guess? Then write the cell yourself, in the empty cell below, a nearest-neighbor rule: embed every training paragraph (`vectors_train`) and every question (`vectors_q`) as in unit 4, give each question the lesson of the training paragraph with the highest cosine, and give each held-out paragraph (`vectors_test`) the lesson of its nearest training paragraph. Put two numbers in the list `answer_2`: the questions it gets right, out of 16, and its accuracy on the held-out paragraphs, rounded to 3 decimals.

**Expected:** `answer_2` is a list of two numbers, a whole number up to 16 and a share between 0 and 1.

In [ ]:
_ = check("Exercise 2", "answer_2", [15, 0.655], tol=5e-4)

<details>
<summary>Hint</summary>

The vectors have length 1, so `vectors_q @ vectors_train.T` holds the cosine of every question with every training paragraph. `argmax(axis=1)` gives the number of the closest one, and `train_lesson` turns it back into a lesson.

</details>

<details>
<summary>Half-solution</summary>

```python
nearest_q = (vectors_q @ vectors_train.T).argmax(axis=1)
...
```

</details>

<details>
<summary>Solution</summary>

```python
nearest_q = (vectors_q @ vectors_train.T).argmax(axis=1)
nearest_t = (vectors_test @ vectors_train.T).argmax(axis=1)
answer_2 = [int(sum(train_lesson[i] == lesson for i, lesson in zip(nearest_q, question_lesson))),
            round(float(np.mean([train_lesson[i] for i in nearest_t] == np.array(test_lesson))), 3)]
```

[15, 0.655]: the nearest training paragraph names the right lesson for 15 of the 16 questions and for 0.655 of the 55 held-out paragraphs (36 of them). The rule fits nothing, and it lands close to the two logistic heads of unit 4: 0.691 on the paragraphs for both, and 15 and 14 of 16 on the questions.

</details>

## Check yourself

### Exercise 3: Check yourself

**Task:** **Spot the flaw** A colleague wants one vector for each lesson, to send a question to the nearest lesson. The code below encodes the whole text of each lesson in one call. Change it so that a lesson's vector is the mean of the vectors of its paragraphs, scaled back to length 1, and put the number of the 16 questions whose nearest lesson vector is the right lesson in `answer_3`.

**Expected:** `answer_3` is a whole number from 0 to 16.

In [ ]:
lesson_vectors = model.encode([" ".join(text for slug, text in paras if slug == lesson) for lesson in LESSONS])
nearest = (vectors_q @ lesson_vectors.T).argmax(axis=1)
answer_3 = int(sum(LESSONS[i] == lesson for i, lesson in zip(nearest, question_lesson)))
# change how lesson_vectors is made: the mean of the paragraph vectors, scaled back to length 1

In [ ]:
_ = check("Exercise 3", "answer_3", 13, traps={11: "That is the whole-lesson text in one call. Look at how many word pieces the model can read in one call."})

<details>
<summary>Hint</summary>

How many word pieces can the model read in one call? Print `model.max_seq_length` and compare it with the number of tokens of a whole lesson.

</details>

<details>
<summary>Half-solution</summary>

```python
vectors_paras = model.encode(texts)
lesson_vectors = ...                                                         # one row for each lesson: the mean of its paragraphs' vectors
lesson_vectors = lesson_vectors / np.linalg.norm(lesson_vectors, axis=1, keepdims=True)
```

</details>

<details>
<summary>Solution</summary>

```python
vectors_paras = model.encode(texts)
lesson_vectors = np.stack([vectors_paras[[i for i, slug in enumerate(labels) if slug == lesson]].mean(axis=0) for lesson in LESSONS])
lesson_vectors = lesson_vectors / np.linalg.norm(lesson_vectors, axis=1, keepdims=True)
nearest = (vectors_q @ lesson_vectors.T).argmax(axis=1)
answer_3 = int(sum(LESSONS[i] == lesson for i, lesson in zip(nearest, question_lesson)))
```

The answer is 13, and the flawed code gets 11. The model card says "By default, input text longer than 256 word pieces is truncated", and each of the eight lessons here holds 982 to 1,496 word pieces, so the vector of each lesson describes only its first paragraphs. The mean of the paragraph vectors uses every paragraph and puts 13 of the 16 questions next to the right lesson.

</details>

## Run all from the top

Restart the session, run every cell from the first, then read this cell last.

In [ ]:
for label in EXERCISES:
    state = RESULTS.get(label)
    print(f"{label}: " + ("correct" if state else "not yet" if state is False else "not started"))
print(f"{sum(1 for label in EXERCISES if RESULTS.get(label))} of {len(EXERCISES)} exercises correct.")